# Học kết hợp (Ensemble Learning) - Biểu quyết đa số (Majority Voting)

## Introduction


Học kết hợp (Ensemble Learning) dựa trên các thuật toán gợi ý sau:
- Lọc dựa trên nội dung - Heuristic
- Lọc dựa trên nội dung - Độ tương đồng nút (Node Similarity)
- Lọc cộng tác - UserKnn với FastRP
- Lọc cộng tác - ItemKnn với FastRP

## Điều kiện tiên quyết

Neo4j server đã được cài đặt phiên bản GDS mới (2.0+).

Thư viện Python `graphdatascience` để vận hành Neo4j GDS.

Truy vấn `Cypher` để tạo các gợi ý.

Gói `py2neo` để ghi ngược pandas dataframe vào cơ sở dữ liệu neo4j.

In [1]:
import os
import textwrap
import configparser

import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import jaccard_score

from py2neo import Graph
from neo4j import GraphDatabase
from graphdatascience import GraphDataScience
from pyvi import ViTokenizer
import torch
from transformers import AutoModel, AutoTokenizer

d:\Thac_Si\De_an_thac_si\code\KG-Rec-Sys-Tourism-SG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Sử dụng file ini cho thông tin đăng nhập, nếu không sử dụng mặc định
HOST = 'neo4j://localhost'
DATABASE = 'neo4j'
PASSWORD = 'password'

NEO4J_CONF_FILE = 'neo4j.ini'

if NEO4J_CONF_FILE is not None and os.path.exists(NEO4J_CONF_FILE):
    config = configparser.RawConfigParser()
    config.read(NEO4J_CONF_FILE)
    HOST = config['NEO4J']['HOST']
    DATABASE = config['NEO4J'].get('DATABASE', 'neo4j')
    USERNAME = config['NEO4J'].get('USERNAME', DATABASE)
    PASSWORD = config['NEO4J']['PASSWORD']
    print(f'Using custom database properties \nHOST: {HOST}; DATABASE: {DATABASE}; PASSWORD: {PASSWORD}')
else:
    print('Could not find database properties file, using defaults')

# Kết nối bằng neo4j python driver
driver = GraphDatabase.driver(HOST, auth=(USERNAME, PASSWORD))

# Connecting with the Neo4j database using GDS library
gds = GraphDataScience(HOST, auth=(USERNAME, PASSWORD))
gds.set_database(DATABASE)

# Connect to Neo4j database using py2neo
graph = Graph(HOST, auth=(USERNAME, PASSWORD), name=DATABASE)

Using custom database properties 
HOST: neo4j://127.0.0.1:7687; DATABASE: neo4j; PASSWORD: 12345678


In [3]:
# Hàm trợ giúp (helper) cho driver
def run(driver, query, params=None):
    with driver.session(database=DATABASE) as session:
        if params is not None:
            return [r for r in session.run(query, params)]
        else:
            return [r for r in session.run(query)]

## 1) Gợi ý Lọc dựa trên nội dung - Phương pháp Heuristic

### Hàm truy vấn

In [ ]:
# HÀM: Tạo gợi ý dựa trên phương pháp Lọc dựa trên nội dung - Heuristic
# INPUT: user_id, poi_id
# OUTPUT: dataframe[user_id, poi_id, rec_poi_id]

def heuristic_recommendation(user_id, poi_id, k=10):
    # Lấy các POI trong cùng khu vực với POI đã được người dùng đánh giá
    records_region = run(driver, textwrap.dedent("""\
        MATCH (user {id: $user_id})-[:REVIEWED]->(poi:Poi {id: $poi_id})-[:LOCATED_AT]->(region:Region)<-[:LOCATED_AT]-(other_poi:Poi)<-[rated:RATED]-(review:Review)
        WHERE poi <> other_poi
        WITH user, poi, other_poi, region, count(DISTINCT rated) AS num_reviews
        RETURN user.id AS user_id, poi.id AS poi_id, other_poi.id AS rec_poi_id, other_poi.name AS rec_poi_name, region.name AS region, num_reviews AS occurrences
        """),
        params = {'user_id': user_id, 'poi_id': poi_id}
    )
    # print(f"Tìm thấy {len(records_region)} records POI có CÙNG KHU VỰC.")
    # Lấy các POI trong cùng danh mục với POI đã được người dùng đánh giá 
    records_category = run(driver, textwrap.dedent("""\
        MATCH (user {id: $user_id})-[:REVIEWED]->(poi:Poi {id: $poi_id})-[:BELONGS_TO]->(category:Category)<-[:BELONGS_TO]-(other_poi:Poi)<-[rated:RATED]-(review:Review)
        WHERE poi <> other_poi
        WITH user, poi, other_poi, category, count(DISTINCT rated) AS num_reviews
        RETURN user.id AS user_id, poi.id AS poi_id, other_poi.id AS rec_poi_id, other_poi.name AS rec_poi_name, category.name AS category_name, num_reviews AS occurrences
        """),
        params = {'user_id': user_id, 'poi_id': poi_id}
    )
    # print(f"Tìm thấy {len(records_category)} records POI có CÙNG DANH MỤC.")
    # Lấy các POI lân cận (NEARBY - trong bán kính 1.5km)
    records_nearby = run(driver, textwrap.dedent("""\
        MATCH (user {id: $user_id})-[:REVIEWED]->(poi:Poi {id: $poi_id})-[n:NEARBY]->(other_poi:Poi)<-[rated:RATED]-(review:Review)
        WHERE poi <> other_poi
        WITH user, poi, other_poi, n.distance_km AS distance, count(DISTINCT rated) AS num_reviews
        RETURN user.id AS user_id, poi.id AS poi_id, other_poi.id AS rec_poi_id, other_poi.name AS rec_poi_name, distance, num_reviews AS occurrences
        """),
        params = {'user_id': user_id, 'poi_id': poi_id}
    )
    # print(f"Tìm thấy {len(records_nearby)} records POI LÂN CẬN (< 1.5km).")
    
    
    # Convert kết quả sang DataFrame và gom nhóm tính trọng số (weight)
    if records_region:
        df_records_region = pd.DataFrame([dict(record) for record in records_region])
        # Group by 'poi_id', 'poi_name', and 'occurrences', then aggregate the count of occurrences
        df_records_region_agg = df_records_region.groupby(['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences']).size().reset_index(name='weight_region')
        # print(f"[Khu vực] Rút gọn còn {len(df_records_region_agg)} records sau khi gom nhóm.")
    else:
        df_records_region_agg = pd.DataFrame(columns=['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences', 'weight_region'])
        # print(f"[Khu vực] Không tìm thấy record nào.")

    if records_category:
        df_records_category = pd.DataFrame([dict(record) for record in records_category])
        # Group by 'poi_id', 'poi_name', and 'occurrences', then aggregate the count of occurrences
        df_records_category_agg = df_records_category.groupby(['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences']).size().reset_index(name='weight_category')
        # print(f" [Danh mục] Rút gọn còn {len(df_records_category_agg)} records (tính trùng lặp thể loại).")
    else:
       df_records_category_agg = pd.DataFrame(columns=['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences', 'weight_category'])
       print(f"[Danh mục] Không tìm thấy record nào.")
    if records_nearby:
        df_nearby = pd.DataFrame([dict(r) for r in records_nearby])
        df_nearby_agg = df_nearby.groupby(['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences']).size().reset_index(name='weight_nearby')
    else:
        df_nearby_agg = pd.DataFrame(columns=['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name', 'occurrences', 'weight_nearby'])
    

    df_records_region_agg.rename(columns={'user_id': 'user_id_region', 'poi_id': 'poi_id_region', 'occurrences': 'occurrences_region'}, inplace=True)
    df_records_category_agg.rename(columns={'user_id': 'user_id_category', 'poi_id': 'poi_id_category', 'occurrences': 'occurrences_category'}, inplace=True)
    df_nearby_agg.rename(columns={'user_id': 'user_id_nearby', 'poi_id': 'poi_id_nearby', 'occurrences': 'occurrences_nearby'}, inplace=True)

    # Tính tần suất xuất hiện của POI trong cả hai danh sách
    # Gộp DataFrame dựa trên 'rec_poi_id'
    recommended_interactions = pd.merge(df_records_region_agg, df_records_category_agg, on=['rec_poi_id', 'rec_poi_name'], suffixes=('_region', '_category'), how='outer')
    recommended_interactions = pd.merge(recommended_interactions, df_nearby_agg, on=['rec_poi_id', 'rec_poi_name'], suffixes=('', '_nearby'), how='outer')
    # print(f"Gộp 3 danh sách (Outer Join): Tổng cộng có {len(recommended_interactions)} records.")

     # Điền giá trị fallback cho user_id, poi_id, occurrences
    recommended_interactions['user_id'] = recommended_interactions['user_id_region'].fillna(recommended_interactions['user_id_category']).fillna(recommended_interactions['user_id_nearby']).fillna(user_id)
    recommended_interactions['poi_id'] = recommended_interactions['poi_id_region'].fillna(recommended_interactions['poi_id_category']).fillna(recommended_interactions['poi_id_nearby']).fillna(poi_id)
    recommended_interactions['occurrences'] = recommended_interactions['occurrences_region'].fillna(recommended_interactions['occurrences_category']).fillna(recommended_interactions['occurrences_nearby']).fillna(0)

    # Điền các giá trị NaN bằng 0 cho các cột 'weight'
    recommended_interactions['weight_region'] = recommended_interactions['weight_region'].fillna(0)
    recommended_interactions['weight_category'] = recommended_interactions['weight_category'].fillna(0)
    recommended_interactions['weight_nearby'] = recommended_interactions['weight_nearby'].fillna(0)

    # print(f"Số record VỪA cùng [khu vực] VỪA cùng [danh mục]: {len(recommended_interactions)}")
    
    # Cộng các cột 'weight' để tính tổng trọng số
    recommended_interactions['total_weight'] = recommended_interactions['weight_region'] + recommended_interactions['weight_category'] + recommended_interactions['weight_nearby']

    # Loại bỏ các cột 'weight' riêng lẻ nếu cần
    recommended_interactions.drop(['user_id_category', 'user_id_nearby', 
                                    'poi_id_category', 'poi_id_nearby', 
                                    'occurrences_category', 'occurrences_nearby', 
                                    'weight_region', 'weight_category', 'weight_nearby'], axis=1, inplace=True, errors='ignore')
    # Sắp xếp DataFrame theo 'total_weight' giảm dần, sau đó theo 'occurrences'
    recommended_interactions = recommended_interactions.sort_values(by=['total_weight', 'occurrences'], ascending=[False, False])

    # Khởi tạo lại chỉ mục cho DataFrame
    recommended_interactions.reset_index(drop=True, inplace=True)
    # Sắp xếp lại các cột
    recommended_interactions = recommended_interactions[['user_id', 'poi_id', 'rec_poi_id', 'rec_poi_name']]
    # Loại bỏ trùng lặp
    recommended_interactions = recommended_interactions.drop_duplicates()

    # Hiển thị DataFrame đã gộp
    return recommended_interactions.head(k)

## 2) Gợi ý Lọc dựa trên nội dung - Độ tương đồng nút

### Chuẩn bị

In [6]:
# Trích xuất dữ liệu thô của nút POI và các thuộc tính của nó từ GDS
result = gds.run_cypher("""
MATCH (poi:Poi)
OPTIONAL MATCH (poi)-[:BELONGS_TO]->(category:Category)
OPTIONAL MATCH (poi)-[:LOCATED_AT]->(region:Region)
RETURN poi.id AS poi_id, 
       poi.name AS name, 
                        
       poi.description AS description, 

       poi.openingHours AS opening_hours, 
       poi.duration AS duration, 
       category.name AS category, 
       region.name AS region,
                        
       poi.price AS price, 
       poi.avgRating AS avg_rating, 
       poi.numReviews AS num_reviews, 
       poi.numReviews_5 AS num_reviews_5, 
       poi.numReviews_4 AS num_reviews_4, 
       poi.numReviews_3 AS num_reviews_3, 
       poi.numReviews_2 AS num_reviews_2, 
       poi.numReviews_1 AS num_reviews_1
""")

# Chuyển đổi kết quả thành DataFrame
df_pois = pd.DataFrame(result)

# Trích xuất poi_id và poi_name duy nhất
df_distinct_pois = df_pois.copy()
df_distinct_pois = df_distinct_pois[['poi_id', 'name']].drop_duplicates()

# Thuộc tính số - Chuẩn hóa Min-Max
# Các thuộc tính: 'avg_rating', 'num_reviews', 'num_reviews_5', 'num_reviews_4', 'num_reviews_3', 'num_reviews_2', 'num_reviews_1'

scaler = MinMaxScaler()
numerical_cols = ['avg_rating', 'num_reviews', 'num_reviews_5', 'num_reviews_4', 'num_reviews_3', 'num_reviews_2', 'num_reviews_1']
# Tạo một DataFrame mới với các cột đã chuẩn hóa và poi_id
df_numerical_cols = df_pois.copy()
df_numerical_cols = df_numerical_cols[['poi_id', 'name'] + numerical_cols]
# chỉ giữ lại các bản ghi duy nhất
df_numerical_cols = df_numerical_cols.drop_duplicates()
# Điền các giá trị thiếu trong cột số bằng 0
df_numerical_cols = df_numerical_cols.fillna(0)

# chuẩn hóa thuộc tính
df_numerical_cols[numerical_cols] = scaler.fit_transform(df_numerical_cols[numerical_cols])


# Thuộc tính phân loại - Mã hóa One-hot
# Các thuộc tính: category, region, opening_Hours, duration

categorical_cols = ['category', 'region', 'opening_hours', 'duration']

# Sao chép df_pois chỉ với các cột phân loại được chỉ định
df_categorical_cols = df_pois.copy()
df_categorical_cols = df_categorical_cols[['poi_id', 'name'] + categorical_cols]

# Thực hiện mã hóa one-hot cho các cột phân loại
df_categorical_cols = pd.get_dummies(df_categorical_cols, columns=categorical_cols)


# Gộp các dòng có cùng poi_id bằng phép toán logic OR
df_categorical_cols = df_categorical_cols.groupby('poi_id').max().reset_index()


# Thuộc tính văn bản - Đếm Token
# Các thuộc tính: description

textual_cols = ['description']

# Sao chép df_pois chỉ với các cột văn bản được chỉ định
df_cols = df_pois.copy()
df_cols = df_cols[['poi_id'] + textual_cols]

# chỉ giữ lại các bản ghi duy nhất
df_cols = df_cols.drop_duplicates()

# điền các giá trị thiếu hoặc rỗng bằng "NULL"
df_cols['description'] = df_cols['description'].fillna('NULL')
empty_description = df_cols['description'].str.strip() == ''
df_cols.loc[empty_description, 'description'] = 'NULL'

# Tải mô tả
descriptions = df_cols['description'].tolist()

# Tokenize tiếng Việt bằng PyVi
segmented_descriptions = [ViTokenizer.tokenize(desc) for desc in descriptions]

# Cấu hình thiết bị GPU/CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Khởi tạo Tokenizer và PhoBERT model
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
phobert = AutoModel.from_pretrained("vinai/phobert-base-v2").to(device)
phobert.eval()

# Hàm trích xuất embedding
def get_phobert_embeddings(texts, batch_size=32):
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            # Tokenize và padding
            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            ).to(device)
            
            # Feed forward
            outputs = phobert(**inputs)
            
            # Sử dụng vector biểu diễn của token [CLS] (ở index 0) làm sentence embedding
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
            
            if (i + batch_size) % (batch_size * 10) == 0 or (i + batch_size) >= len(texts):
                print(f"Đã xử lý {min(i + batch_size, len(texts))}/{len(texts)} mô tả...")
                
    return np.vstack(embeddings)

# Trích xuất embedding
embeddings = get_phobert_embeddings(segmented_descriptions)
# Tạo DataFrame lưu trữ vector embedding (768 chiều)
df_textual_cols = pd.DataFrame(embeddings)
# Đặt lại cột poi_id làm cột đầu tiên
df_textual_cols.insert(0, 'poi_id', df_cols['poi_id'].values)

print(f"Hoàn thành! Kích thước ma trận embedding: {df_textual_cols.shape}")
df_textual_cols

# Tính toán độ tương đồng giữa các cặp POI

# Chuẩn bị dữ liệu dạng dictionary để tìm kiếm O(1) và loại bỏ overhead của Pandas
categorical_dict = df_categorical_cols.set_index('poi_id').drop(columns=['name'], errors='ignore').to_dict(orient='index')
categorical_dict = {k: np.array(list(v.values()), dtype=bool) for k, v in categorical_dict.items()}

numerical_dict = df_numerical_cols.set_index('poi_id').drop(columns=['name'], errors='ignore').to_dict(orient='index')
numerical_dict = {k: np.array(list(v.values()), dtype=float) for k, v in numerical_dict.items()}

# Trích xuất và tiền chuẩn hóa vector L2 cho độ tương đồng Cosine nhanh
textual_dict = df_textual_cols.set_index('poi_id').to_dict(orient='index')
textual_dict_norm = {}
for k, v in textual_dict.items():
    arr = np.array(list(v.values()), dtype=float)
    norm = np.linalg.norm(arr)
    textual_dict_norm[k] = arr / norm if norm > 0 else arr

num_cat_cols = len(categorical_cols) # 3
num_num_cols = len(numerical_cols) # 8
num_text_cols = len(textual_cols) # 1
total_weights = num_cat_cols + num_num_cols + num_text_cols

poi_ids = df_distinct_pois['poi_id'].tolist()
n_pois = len(poi_ids)

print(f"Bắt đầu tính toán độ tương đồng cho {n_pois * (n_pois - 1) // 2} cặp...")

# Khởi tạo ma trận tương đồng
similarity_pairs = []

# Vòng lặp tối ưu hóa bằng NumPy
for i in range(n_pois):
    poi1_id = poi_ids[i]
    
    # Lấy sẵn dữ liệu của poi1
    p1_cat = categorical_dict[poi1_id]
    p1_num = numerical_dict[poi1_id]
    p1_text = textual_dict_norm[poi1_id]
    
    for j in range(i + 1, n_pois):
        poi2_id = poi_ids[j]
        
        p2_cat = categorical_dict[poi2_id]
        p2_num = numerical_dict[poi2_id]
        p2_text = textual_dict_norm[poi2_id]
        
        # 1. Jaccard Similarity cho thuộc tính phân loại
        intersection = np.sum(p1_cat & p2_cat)
        union = np.sum(p1_cat | p2_cat)
        cat_cols_similarity = intersection / union if union != 0 else 0.0
        
        # 2. Euclidean Similarity cho thuộc tính số
        euclidean_distance = np.linalg.norm(p1_num - p2_num)
        num_cols_similarity = 1 / (1 + euclidean_distance)
        
        # 3. Cosine Similarity cho thuộc tính văn bản (đã chuẩn hóa L2 nên chỉ cần nhân vô hướng)
        text_cols_similarity = np.dot(p1_text, p2_text)
        
        # 4. Trọng số tổng hợp
        similarity = (num_cat_cols * cat_cols_similarity + 
                      num_num_cols * num_cols_similarity + 
                      num_text_cols * text_cols_similarity) / total_weights
        
        # Chỉ lưu các cặp có độ tương đồng lớn hơn hoặc bằng 0.5
        if similarity >= 0.5:
            similarity_pairs.append((poi1_id, poi2_id, similarity))

# Tạo DataFrame kết quả
df_similarity = pd.DataFrame(similarity_pairs, columns=['poi1_id', 'poi2_id', 'Similarity'])

# Sắp xếp theo mức độ tương đồng giảm dần
df_similarity = df_similarity.sort_values(by='Similarity', ascending=False).reset_index(drop=True)

print(f"Số lượng cặp POI tương đồng (>= 0.5): {len(df_similarity)}")

# ghi mối quan hệ SIMILAR giữa các POI với thuộc tính similarity

# # Lặp qua các dòng của DataFrame và ghi mối quan hệ vào Neo4j
# for index, row in df_similarity.iterrows():
#     poi1_id = row['poi1_id']
#     poi2_id = row['poi2_id']
#     similarity = row['Similarity']
    
#     # Ghi mối quan hệ vô hướng giữa poi1_id và poi2_id kèm thuộc tính similarity
#     query = f"""
#     MATCH (poi1:Poi {{id: {poi1_id}}})
#     MATCH (poi2:Poi {{id: {poi2_id}}})
#     MERGE (poi1)-[s1:CBF_SIMILAR]->(poi2)
#     ON CREATE SET s1.score = {similarity}
#     MERGE (poi1)<-[s2:CBF_SIMILAR]-(poi2)
#     ON CREATE SET s2.score = {similarity}
#     """
#     graph.run(query)
    
# print("Hoàn thành ghi toàn bộ dữ liệu quan hệ vào Neo4j!")

# Chuyển DataFrame thành list of dicts và lưu mối quan hệ vào Neo4j
# Đã lưu quan hệ vào Neo4j thì không chạy lại cell này nữa !!!
data_list = df_similarity.to_dict(orient='records')

batch_query = """
UNWIND $rows AS row
MATCH (poi1:Poi {id: row.poi1_id})
MATCH (poi2:Poi {id: row.poi2_id})
MERGE (poi1)-[s1:CBF_SIMILAR]->(poi2)
ON CREATE SET s1.score = row.Similarity
MERGE (poi1)<-[s2:CBF_SIMILAR]-(poi2)
ON CREATE SET s2.score = row.Similarity
"""

batch_size = 500000
total_rows = len(data_list)

for i in range(0, total_rows, batch_size):
    batch = data_list[i : i + batch_size]
    graph.run(batch_query, rows=batch)
    print(f"Đã ghi xong từ {i} đến {min(i + batch_size, total_rows)}")

print("Hoàn thành ghi toàn bộ dữ liệu quan hệ vào Neo4j!")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 63530.52it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Đã xử lý 320/3326 mô tả...
Đã xử lý 640/3326 mô tả...
Đã xử lý 960/3326 mô tả...
Đã xử lý 1280/3326 mô tả...
Đã xử lý 1600/3326 mô tả...
Đã xử lý 1920/3326 mô tả...
Đã xử lý 2240/3326 mô tả...
Đã xử lý 2560/3326 mô tả...
Đã xử lý 2880/3326 mô tả...
Đã xử lý 3200/3326 mô tả...
Đã xử lý 3326/3326 mô tả...
Hoàn thành! Kích thước ma trận embedding: (3326, 769)
Bắt đầu tính toán độ tương đồng cho 5529475 cặp...
Số lượng cặp POI tương đồng (>= 0.5): 3293643
Đã ghi xong từ 0 đến 500000
Đã ghi xong từ 500000 đến 1000000
Đã ghi xong từ 1000000 đến 1500000
Đã ghi xong từ 1500000 đến 2000000
Đã ghi xong từ 2000000 đến 2500000
Đã ghi xong từ 2500000 đến 3000000
Đã ghi xong từ 3000000 đến 3293643
Hoàn thành ghi toàn bộ dữ liệu quan hệ vào Neo4j!


### Hàm truy vấn

In [7]:
# HÀM: Tạo gợi ý dựa trên phương pháp Lọc dựa trên nội dung - Độ tương đồng nút (Node Similarity)
# INPUT: poi_id
# OUTPUT: dataframe[poi_id, rec_poi_id]

def similar_poi_recommendation(poi_id):
    result = gds.run_cypher(
        """
            MATCH (p1:Poi {id: $target_poi})-[s:CBF_SIMILAR]->(p2:Poi)
            RETURN p1.id as poi_id, p2.id as rec_poi_id
            ORDER BY s.score DESC
        """, params = {'target_poi': poi_id}
    )
    result = result.drop_duplicates()
    return result

## 3) Gợi ý Lọc cộng tác - User-Based KNN dựa trên vector nhúng FastRP

### Chuẩn bị

Đồ thị được chiếu và vector nhúng FastRP này sẽ được sử dụng cho cả thuật toán (3) và (4).

In [8]:
# Kiểm tra và xóa đồ thị 'myGraph' đã tồn tại trong bộ nhớ trước khi chiếu
if gds.graph.exists("myGraph")["exists"]:
    gds.graph.drop(gds.graph.get("myGraph"))

# Đồ thị chiếu (Projection Graph)

# định nghĩa cách chiếu cơ sở dữ liệu vào GDS
node_projection = ["User", "Poi"]
relationship_projection = {"REVIEWED": {"orientation": "UNDIRECTED", "properties": "rating"}}

# tiến hành chiếu đồ thị
G, result = gds.graph.project("myGraph", node_projection, relationship_projection)


# Tạo các vector nhúng FastRP

# chạy FastRP và đột biến (mutate) đồ thị được chiếu của chúng ta với kết quả
result = gds.fastRP.mutate(
    G,
    randomSeed=42,
    embeddingDimension=256,
    relationshipWeightProperty="rating",
    iterationWeights=[0, 1, 1, 1],
    mutateProperty="embedding"
)

print(f"Số lượng embedding vectors được tạo ra: {result['nodePropertiesWritten']}")

Số lượng embedding vectors được tạo ra: 24641


In [9]:
# Tính tương đồng với User-based KNN

# Chạy KNN với siêu tham số topK tối ưu và ghi ngược lại DB

topK_best = 9

result = gds.knn.write(
    G,
    topK=topK_best,
    nodeLabels=['User'],
    nodeProperties=["embedding"],
    randomSeed=42,
    concurrency=1,
    sampleRate=1.0,
    deltaThreshold=0.0,
    writeRelationshipType="CF_SIMILAR_USER",
    writeProperty="score",

)

print(f"Relationships produced: {result['relationshipsWritten']}")
print(f"Nodes compared: {result['nodesCompared']}")
print(f"Mean similarity: {result['similarityDistribution']['mean']}")

 Knn: 100%|██████████| 100.0/100 [00:18<00:00,  5.41%/s, status: FINISHED]                                

Relationships produced: 191835
Nodes compared: 21315
Mean similarity: 0.9614082358896174


### Hàm truy vấn

In [10]:
# HÀM: Tạo gợi ý dựa trên Lọc cộng tác - User-based KNN dựa trên vector nhúng FastRP
# INPUT: user_id
# OUTPUT: dataframe[user_id, rec_poi_id]

def userKNN_recommendation(user_id):

    result = gds.run_cypher(
        """
            MATCH (u1:User {id: $target_user})-[s:CF_SIMILAR_USER]->(u2:User)-[:REVIEWED]->(p:Poi)
            WITH u1, p, s.score AS user_similarity
            RETURN u1.id as user_id, p.id as rec_poi_id
            ORDER BY user_similarity DESC, p.avgRating DESC
        """, params = {'target_user': user_id}
    )
    result = result.drop_duplicates()
    return result

## 4) Gợi ý Lọc cộng tác - Item-Based KNN dựa trên vector nhúng FastRP

### Chuẩn bị

In [11]:
# Tính tương đồng với Item-based KNN

# Chạy KNN với siêu tham số topK tối ưu và ghi ngược lại DB

topK_best = 1

result = gds.knn.write(
    G,
    topK=topK_best,
    nodeLabels = ['Poi'],
    nodeProperties=["embedding"],
    randomSeed=42,
    concurrency=1,
    sampleRate=1.0,
    deltaThreshold=0.0,
    similarityCutoff = 0.5,
    writeRelationshipType="CF_SIMILAR_POI",
    writeProperty="score"
)

print(f"Relationships produced: {result['relationshipsWritten']}")
print(f"Nodes compared: {result['nodesCompared']}")
print(f"Mean similarity: {result['similarityDistribution']['mean']}")

 Knn: 100%|██████████| 100.0/100 [00:00<00:00, 88.94%/s, status: FINISHED]                                

Relationships produced: 1340
Nodes compared: 3326
Mean similarity: 0.6156053457687151


### Hàm truy vấn

In [12]:
# HÀM: Tạo gợi ý dựa trên Lọc cộng tác - Item-Based KNN dựa trên vector nhúng FastRP
# INPUT: poi_id
# OUTPUT: dataframe[poi_id, rec_poi_id]

def itemKNN_recommendation(poi_id):
    result = gds.run_cypher(
        """
            MATCH (p1:Poi {id: $target_poi})-[s:CF_SIMILAR_POI]->(p2:Poi)
            RETURN p1.id as poi_id, p2.id as rec_poi_id
            ORDER BY s.score DESC, p2.avgRating DESC
        """, params = {'target_poi': poi_id}
    )
    result = result.drop_duplicates()
    return result

## Tiến hành gợi ý bằng Học kết hợp (Ensemble Learning)

Tạo gợi ý POI cho người dùng từ các mô hình kết hợp sử dụng biểu quyết đa số.

In [13]:
# HÀM: dọn dẹp và chuẩn bị từng dataframe sau khi gọi thuật toán gợi ý, chuẩn bị cho học kết hợp
def df_cleaning (df):
    if not df.empty:    # Reset index, get rank, and re-arrange columns
        df.reset_index(drop=True, inplace=True)          
        df = df.reset_index().rename(columns={'index': 'rank'})
        df['rank'] += 1
        df = df.reindex(columns=['user_id', 'poi_id', 'rec_poi_id', 'rank', 'df_name'])

    return df

In [14]:
# HÀM: Tạo gợi ý dựa trên Học kết hợp - Biểu quyết đa số (Majority Voting)
# ĐẦU VÀO: poi_id, user_id, algo_combination
# ĐẦU RA: dataframe[poi_id, user_id, rec_poi_id]

def ensemble_recommendation(poi_id, user_id, algo_combination):

    # Dựa trên tổ hợp thuật toán đã chọn, quyết định có gọi từng hàm gợi ý tương ứng hay không

    if 1 in algo_combination:
        rec_CBF_heuristic = heuristic_recommendation(user_id, poi_id)   # OUTPUT: dataframe[user_id, poi_id, rec_poi_id]
        rec_CBF_heuristic['df_name'] = 'rec_CBF_heuristic'              # Add DataFrame name as a column
        rec_CBF_heuristic = df_cleaning (rec_CBF_heuristic)             # Clean up df for ensemble
    else:
        rec_CBF_heuristic = pd.DataFrame()

    if 2 in algo_combination:
        rec_CBF_similarity = similar_poi_recommendation(poi_id)         # OUTPUT: dataframe[poi_id, rec_poi_id]
        rec_CBF_similarity['df_name'] = 'rec_CBF_similarity'            # Add DataFrame name as a column
        rec_CBF_similarity['user_id'] = user_id                         # Add missing columns
        rec_CBF_similarity = df_cleaning (rec_CBF_similarity)           # Clean up df for ensemble
    else:
        rec_CBF_similarity = pd.DataFrame()

    if 3 in algo_combination:
        rec_CF_userKnn =  userKNN_recommendation(user_id)               # OUTPUT: dataframe[user_id, rec_poi_id]
        rec_CF_userKnn['df_name'] = 'rec_CF_userKnn'                    # Add DataFrame name as a column
        rec_CF_userKnn['poi_id'] = poi_id                               # Add missing columns
        rec_CF_userKnn = df_cleaning (rec_CF_userKnn)                   # Clean up df for ensemble
    else:
        rec_CF_userKnn = pd.DataFrame()

    if 4 in algo_combination:
        rec_CF_itemKnn = itemKNN_recommendation(poi_id)                 # OUTPUT: dataframe[poi_id, rec_poi_id]
        rec_CF_itemKnn['df_name'] = 'rec_CF_itemKnn'                    # Add DataFrame name as a column
        rec_CF_itemKnn['user_id'] = user_id                             # Add missing columns
        rec_CF_itemKnn = df_cleaning (rec_CF_itemKnn)                   # Clean up df for ensemble
    else:
        rec_CF_itemKnn = pd.DataFrame()

    # In ra các DataFrame đã được sắp xếp lại
    # print(rec_CBF_heuristic)
    # print(rec_CBF_similarity)
    # print(rec_CF_userKnn)
    # print(rec_CF_itemKnn)
    
    # Nối các DataFrame theo các dòng
    merged_df = pd.concat([rec_CF_itemKnn, rec_CF_userKnn, rec_CBF_similarity, rec_CBF_heuristic])
    # print(f'merged_df: \n{merged_df}')

    # kiểm tra xem merged_df có rỗng không
    if not merged_df.empty:
        # Nhóm theo user_id, poi_id, rec_poi_id và tính toán thứ hạng trung bình (average rank) và số lượng
        grouped_df = merged_df.groupby(['user_id', 'poi_id', 'rec_poi_id']).agg({'rank': 'mean', 'df_name': 'count'}).reset_index()

        # Đổi tên cột đếm thành count
        grouped_df.rename(columns={'df_name': 'count'}, inplace=True)

        # loại bỏ các mục có count = 1
        grouped_df = grouped_df[grouped_df['count'] > 1]

        # Sắp xếp theo count giảm dần và thứ hạng trung bình tăng dần
        sorted_df = grouped_df.sort_values(by=['count', 'rank'], ascending=[False, True])
        #print(f'sorted_df: \n{sorted_df}')

        # Xóa cột 'count' và 'rank'
        result = sorted_df.drop(columns=['count', 'rank'])
        #result = sorted_df.copy()
        result.reset_index(drop=True, inplace=True)
    else:
        result = merged_df.drop(columns=['df_name'])

    return result


# ID của người dùng mục tiêu
user_id = 1322
# ID của POI mục tiêu
poi_id = 4552853

# tổ hợp thuật toán cho học kết hợp
algo_combination = [1,2,3,4]

# Chọn từ các thuật toán bên dưới:
#(1) Lọc dựa trên nội dung - Heuristic
#(2) Lọc dựa trên nội dung - Độ tương đồng nút (Node Similarity)
#(3) Lọc cộng tác - UserKnn với FastRP
#(4) Lọc cộng tác - ItemKnn với FastRP

ensemble_recommendation(poi_id, user_id, algo_combination)

Tìm thấy 0 records POI có CÙNG KHU VỰC.
Tìm thấy 0 records POI có CÙNG DANH MỤC.
Tìm thấy 0 records POI LÂN CẬN (< 1.5km).
[Danh mục] Không tìm thấy record nào.


,user_id,poi_id,rec_poi_id
0,1322,4552853,2475475
1,1322,4552853,8630608


# Đánh giá

In [15]:
# DataFrame các POI
df_pois = gds.run_cypher("""\
    MATCH (poi:Poi)    
    RETURN poi.id
    """)

df_pois

,poi.id
0,311103
1,2005826
2,311087
3,4542125
4,311089
...,...
3321,34509929
3322,34508995
3323,34515006
3324,34435391


In [16]:
# DataFrame các đánh giá

df_reviews = gds.run_cypher("""\
    MATCH (user:User)-[review:REVIEWED]->(poi:Poi)
    RETURN user.id AS user_id, poi.id AS poi_id
    """)

df_reviews

,user_id,poi_id
0,1,311103
1,2,311103
2,3,311103
3,4,311103
4,5,311103
...,...,...
23943,21311,15129760
23944,21312,15129760
23945,21313,15129760
23946,21314,15129760


In [17]:
# Nhóm theo 'user_id' và đếm số lần xuất hiện
user_counts = df_reviews.groupby('user_id').size()

# Lọc ra các người dùng có ít hơn 5 lần xuất hiện
valid_users = user_counts[user_counts >= 5].index

# Lọc DataFrame gốc dựa trên danh sách người dùng hợp lệ
filtered_df_reviews = df_reviews[df_reviews['user_id'].isin(valid_users)].copy()
filtered_df_reviews

,user_id,poi_id
50,51,311103
51,52,311103
103,104,311103
116,117,311103
118,119,311103
...,...,...
23570,481,12829864
23571,1303,12829864
23573,2886,12829864
23613,1246,9974597


In [18]:
# Chia tập dữ liệu thành 90% tập huấn luyện (training) và 10% tập kiểm tra (test)
df_train, df_test = train_test_split(filtered_df_reviews, test_size=0.1, random_state=100)

df_train

,user_id,poi_id
23463,181,9806390
2694,473,311094
3279,1177,1910195
2331,1334,10836601
8110,51,16657089
...,...,...
4403,284,311092
555,493,4542125
2215,1459,4552853
745,52,2037764


In [19]:
df_test

,user_id,poi_id
3284,1253,1910195
5049,683,552637
14873,606,10044301
1854,472,317896
2336,1415,10836601
...,...,...
1378,1006,10005057
23298,181,10388517
23338,474,1809055
545,484,311087


In [20]:
# Trích xuất các tương tác thực tế của tất cả các cặp POI được đánh giá bởi cùng người dùng

# Nhóm theo user_id và tổng hợp các poi_id thành một danh sách
grouped = df_reviews.groupby('user_id')['poi_id'].apply(list)

# Khởi tạo một DataFrame trống cho kết quả
df_true_interactions = pd.DataFrame(columns=['user_id', 'poi_id', 'rec_poi_id'])

# Lặp qua từng nhóm
for user_id, poi_ids in grouped.items():
    # Tạo cặp target_poi_id và poi_id cho mỗi người dùng
    entry = [(user_id, poi_id, other_poi_id) for poi_id in poi_ids for other_poi_id in poi_ids if poi_id != other_poi_id]
    df_entry = pd.DataFrame(entry, columns=['user_id', 'poi_id', 'rec_poi_id'])
    # Thêm các cặp vào DataFrame kết quả
    df_true_interactions = pd.concat([df_true_interactions, df_entry], ignore_index=True)

df_true_interactions = df_true_interactions.drop_duplicates()
# Hiển thị DataFrame kết quả
df_true_interactions

,user_id,poi_id,rec_poi_id
0,26,311103,2005826
1,26,311103,311087
2,26,311103,8587831
3,26,2005826,311103
4,26,2005826,311087
...,...,...,...
19093,20779,10836471,13613302
19094,20920,33018545,33012832
19095,20920,33012832,33018545
19096,20985,27752784,33012832


In [21]:
# Lấy tất cả các thực thể liên quan bằng cách gộp tương tác thực tế và thực thể kiểm tra theo user id và poi id
df_all_relevant = pd.merge(df_test, df_true_interactions, on=['user_id', 'poi_id'], how='inner')
# kiểm tra cặp POI không phải là cùng một POI
df_all_relevant = df_all_relevant[df_all_relevant['poi_id'] != df_all_relevant['rec_poi_id']]
# Loại bỏ trùng lặp
df_all_relevant = df_all_relevant.drop_duplicates()

df_all_relevant

,user_id,poi_id,rec_poi_id
0,1253,1910195,10005057
1,1253,1910195,317893
2,1253,1910195,10836601
3,1253,1910195,1830324
4,1253,1910195,550709
...,...,...,...
1443,313,8290115,317896
1444,313,8290115,454974
1445,313,8290115,317893
1446,313,8290115,2414430


# Bắt đầu chạy thuật toán để lấy kết quả gợi ý

algo [1,2,3,4]

algo [2,3,4]

algo [1,3,4]

algo [1,2,4]

algo [1,2,3]

algo [1,2]

algo [1,3]

algo [1,4]

algo [2,3]

algo [2,4]

algo [3,4]

In [82]:
# Lấy gợi ý cho từng dòng trong tập kiểm tra

# Tinh chỉnh lựa chọn tổ hợp thuật toán để đạt kết quả tốt hơn
algo_combination = [3,4]

df_all_retrieved = pd.DataFrame()
for index, row in df_test.iterrows():
    user_id = row['user_id']
    poi_id = row['poi_id']
    #print(f'\nuser_id: {user_id}')
    #print(f'poi_id: {poi_id}')

    recommended_interactions = ensemble_recommendation(poi_id, user_id, algo_combination)

    # Nối các tương tác gợi ý với test_recommendations
    df_all_retrieved = pd.concat([df_all_retrieved, recommended_interactions], ignore_index=True)

# Loại bỏ trùng lặp
df_all_retrieved = df_all_retrieved.drop_duplicates()

df_all_retrieved

,user_id,poi_id,rec_poi_id
0,1229,10085046,4498781
1,1055,317893,2273343
2,484,7109031,2414430
3,189,5505885,2273343
4,1038,5505885,2273343
5,711,2407915,1910195
6,711,9582552,2273343
7,1221,9806454,8310942
8,313,317893,2273343
9,1004,9582552,2273343


# Số lượng thực thể được lấy ra:

algo [1,2,3,4]: 3347 - 2482

algo [2,3,4]: 2610 - 1777

algo [1,3,4]: 328 - 166

algo [1,2,4]: 1147 - 830

algo [1,2,3]: 3265 - 2421

algo [1,2]: 1050 - 754

algo [1,3]: 310 - 143

algo [1,4]: 8 - 9

algo [2,3]: 2525 - 1710

algo [2,4]: 105 - 85

algo [3,4]: 20 - 20

In [83]:
# Get all relevant retrieved instance by merging the relevant and recommended interactions
df_retrived_relevant = pd.merge(df_all_relevant, df_all_retrieved, on=['user_id', 'poi_id', 'rec_poi_id'], how='inner')
df_retrived_relevant

,user_id,poi_id,rec_poi_id
0,1055,317893,2273343
1,711,2407915,1910195
2,711,9582552,2273343
3,1004,9582552,2273343
4,181,4474246,8295665
5,2129,9996374,7927054
6,474,1809055,311087


# Số lượng bản ghi liên quan được lấy ra (True Positive):

algo [1,2,3,4]: 999 - 632

algo [2,3,4]: 944 - 587

algo [1,3,4]: 145 - 68

algo [1,2,4]: 205 - 75

algo [1,2,3]: 994 - 626

algo [1,2]: 193 - 64

algo [1,3]: 138 - 60

algo [1,4]: 1 - 1

algo [2,3]: 939 - 580

algo [2,4]: 13 - 12

algo [3,4]: 8 - 7

In [84]:
# Tính chỉ số độ chính xác (precision score)
relevant_retrieved = df_retrived_relevant.shape[0]
all_retrived = df_all_retrieved.shape[0]

precision = relevant_retrieved / all_retrived

print(f'Chỉ số độ chính xác (Precision Score): {precision}')

Chỉ số độ chính xác (Precision Score): 0.35


# Chỉ số độ chính xác (Precision Score) 

algo [1,2,3,4]: 0.29847624738571854 - 0.25463336019339244

algo [2,3,4]: 0.36168582375478925 - 0.33033202025886327

algo [1,3,4]: 0.4420731707317073 - 0.40963855421686746

algo [1,2,4]: 0.17872711421098517 - 0.09036144578313253

algo [1,2,3]: 0.30444104134762634 - 0.25857083849648904

algo [1,2]: 0.1838095238095238 - 0.08488063660477453

algo [1,3]: 0.44516129032258067 - 0.4195804195804196

algo [1,4]: 0.125 - 0.1111111111111111

algo [2,3]: 0.3718811881188119 - 0.3391812865497076

algo [2,4]: 0.12380952380952381 - 0.1411764705882353

algo [3,4]: 0.4 - 0.35

In [85]:
# Tính chỉ số độ bao phủ (recall score)
relevant_retrieved = df_retrived_relevant.shape[0]
all_relevant = df_all_relevant.shape[0]

recall = relevant_retrieved / all_relevant
print(f'Chỉ số độ bao phủ (Recall Score): {recall}')

Chỉ số độ bao phủ (Recall Score): 0.004834254143646409


# Chỉ số độ bao phủ (Recall Score) 

algo [1,2,3,4]: 0.5294117647058824 - 0.43646408839779005

algo [2,3,4]: 0.5002649708532061 - 0.4053867403314917

algo [1,3,4]: 0.07684154742978272 - 0.04696132596685083

algo [1,2,4]: 0.1086380498145204 - 0.05179558011049724

algo [1,2,3]: 0.5267620561738209 - 0.43232044198895025

algo [1,2]: 0.10227874933757286 - 0.04419889502762431

algo [1,3]: 0.07313195548489666 - 0.04143646408839779

algo [1,4]: 0.0005299417064122947 - 0.0006906077348066298

algo [2,3]: 0.49761526232114467 - 0.4005524861878453

algo [2,4]: 0.00688924218335983 - 0.008287292817679558

algo [3,4]: 0.0042395336512983575 - 0.004834254143646409

In [86]:
# Tính chỉ số độ phủ (coverage score)
num_recommended_pois = df_all_retrieved['rec_poi_id'].nunique()
num_all_pois = df_pois.shape[0]

coverage = num_recommended_pois / num_all_pois
print(f'Chỉ số độ phủ (Coverage Score): {coverage}')

Chỉ số độ phủ (Coverage Score): 0.0036079374624173183


# Chỉ số độ phủ (Coverage Score) 

algo [1,2,3,4]: 0.10614101592115238 - 0.08208057726999399

algo [2,3,4]: 0.06974981046247157 - 0.04359591100420926

algo [1,3,4]: 0.0200909780136467 - 0.0132291040288635

algo [1,2,4]: 0.07922668688400303 - 0.06434155141310884

algo [1,2,3]: 0.09742228961334344 - 0.07696933253156946

algo [1,2]: 0.06557998483699773 - 0.05472038484666266

algo [1,3]: 0.017437452615617893 - 0.010523150932050512

algo [1,4]: 0.0018953752843062926 - 0.0003006614552014432

algo [2,3]: 0.060652009097801364 - 0.03788334335538184

algo [2,4]: 0.019332827899924184 - 0.012627781118460614

algo [3,4]: 0.006065200909780136 - 0.0036079374624173183

In [87]:
# Tính điểm F1 (F1 score)
f1 = (2 * precision * recall) / (precision + recall)
print(f'F1 Score: {f1}')

F1 Score: 0.009536784741144413


# Điểm F1 (F1 Score) 

algo [1,2,3,4]: 0.38173481085212074 - 0.32162849872773536

algo [2,3,4]: 0.41983544585279065 - 0.364031007751938

algo [1,3,4]: 0.1309255079006772 - 0.08426270136307311

algo [1,2,4]: 0.13513513513513514 - 0.06584723441615452

algo [1,2,3]: 0.38586956521739124 - 0.32359782889635563

algo [1,2]: 0.13142662580864828 - 0.058128973660308815

algo [1,3]: 0.12562585343650431 - 0.07542426147077311

algo [1,4]: 0.0010554089709762533 - 0.0013726835964310226

algo [2,3]: 0.4256572982774252 - 0.36732108929702345

algo [2,4]: 0.013052208835341366 - 0.015655577299412915

algo [3,4]: 0.008390141583639224 - 0.009536784741144413

## Dọn dẹp tài nguyên

Xóa cả trạng thái in-memory của GDS và cơ sở dữ liệu.

In [88]:
# Xóa đồ thị chiếu của chúng ta khỏi danh mục GDS
G.drop()

# Xóa tất cả dữ liệu ví dụ khỏi cơ sở dữ liệu
# _ = gds.run_cypher("MATCH (n) DETACH DELETE n")

graphName                                                          myGraph
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                            24641
relationshipCount                                                    47896
configuration            {'relationshipProjection': {'REVIEWED': {'orie...
density                                                           0.000079
creationTime                           2026-07-22T21:19:03.115149500+07:00
modificationTime                       2026-07-22T21:19:03.529436300+07:00
schema                   {'relationships': {'REVIEWED': {'rating': 'Flo...
schemaWithOrientation    {'relationships': {'REVIEWED': {'properties': ...
Name: 0, dtype: object